In [1]:
# %% 셀 1: 데이터 로드 + 임베딩 로드 + STT 로드 + train/eval 분리
import os, json, random
import numpy as np
import torch
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
CELL_SIZE_X = 9
CELL_SIZE_Y = 16
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


# ── STT 세그먼트 복원 함수 ──
def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None

    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])

        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f

    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


# ── 단일 영상 로드 함수 ──
def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    if not instances:
        return None

    duration = data.get("duration", max(inst["end_sec"] for inst in instances))

    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))

        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })

    # STT 로드
    video_name = os.path.basename(path)[:-5]  # .json 제거
    stt_path = os.path.join(STT_DIR, channel, video_name + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass

    return {
        "channel": channel,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


# ── 병렬 로드 ──
json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

print(f"📁 파일 수: {len(json_paths):,}개")

channel_set = set()
samples = []

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        if result is not None:
            channel_set.add(result["channel"])
            samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# ── STT 통계 ──
stt_counts = [len(s["stt_segments"]) for s in samples]
stt_counts = np.array(stt_counts)
print(f"\n📊 영상별 STT 세그먼트 수")
print(f"  mean: {stt_counts.mean():.1f}  median: {np.median(stt_counts):.0f}  "
      f"p99: {np.percentile(stt_counts, 99):.0f}  max: {stt_counts.max()}")

# ── train/eval 분리 ──
rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples, eval_samples = [], []
for ch, ch_samples in by_channel.items():
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

inst_counts = np.array([len(s["instances"]) for s in samples])
print(f"\n✅ 영상: {len(samples):,}개  |  채널: {len(channels)}개")
print(f"✅ train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")
print(f"📊 인스턴스 수 — mean: {inst_counts.mean():.0f}  p99: {np.percentile(inst_counts, 99):.0f}  max: {inst_counts.max()}")

✅ 임베딩 로드: 2,101,956개  |  dim=1024
📁 파일 수: 66,400개


로드: 100%|██████████| 66400/66400 [00:13<00:00, 5059.09it/s]



📊 영상별 STT 세그먼트 수
  mean: 13.7  median: 11  p99: 53  max: 151

✅ 영상: 59,876개  |  채널: 664개
✅ train: 56,556  |  eval: 3,320
📊 인스턴스 수 — mean: 62  p99: 419  max: 4251


In [2]:
# %% 셀 2: Dataset + DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 32
BIN_SIZE = 0.1
MAX_T = 2000
K_TELOP = 20   # 프레임별 최대 동시 텔롭 슬롯 (p99=18)
K_STT = 5      # 프레임별 최대 동시 STT 슬롯
SLOT_DIM = 128  # 임베딩 투영 차원


class FrameMaskDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = samples
        self.channel2id = channel2id
        self.text2emb = text2emb

        # 고정 random projection (1024 → SLOT_DIM)
        self.telop_proj = nn.Linear(EMB_DIM, SLOT_DIM, bias=False)
        nn.init.orthogonal_(self.telop_proj.weight)
        self.telop_proj.requires_grad_(False)

        self.stt_proj = nn.Linear(EMB_DIM, SLOT_DIM, bias=False)
        nn.init.orthogonal_(self.stt_proj.weight)
        self.stt_proj.requires_grad_(False)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        duration = max(s["duration"], 0.1)
        T = min(int(np.ceil(duration / BIN_SIZE)), MAX_T)

        channel_id = self.channel2id.get(s["channel"], 0)
        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)

        instances = s["instances"]
        n_inst = len(instances)

        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends = np.array([inst["end"] for inst in instances])
        inst_tlens = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances], dtype=np.float32)
        inst_cy = np.array([inst["y"] for inst in instances], dtype=np.float32)
        inst_w = np.array([inst["w"] for inst in instances], dtype=np.float32)
        inst_h = np.array([inst["h"] for inst in instances], dtype=np.float32)

        # 인스턴스별 Qwen 임베딩 → proj
        with torch.no_grad():
            inst_embs_full = torch.stack([
                self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances
            ])
            inst_embs_proj = self.telop_proj(inst_embs_full)  # (n_inst, SLOT_DIM)

        # STT 세그먼트
        stt_segments = s["stt_segments"]
        n_stt = len(stt_segments)
        if n_stt > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends = np.array([seg["end"] for seg in stt_segments])
            with torch.no_grad():
                stt_embs_full = torch.stack([
                    self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments
                ])
                stt_embs_proj = self.stt_proj(stt_embs_full)  # (n_stt, SLOT_DIM)
        else:
            stt_starts = np.array([])
            stt_ends = np.array([])
            stt_embs_proj = torch.zeros(0, SLOT_DIM)

        # ── 프레임별 ──
        scalar_feats = np.zeros((T, 3), dtype=np.float32)  # time, n_active, n_stt_active
        telop_slots = torch.zeros(T, K_TELOP, SLOT_DIM)
        telop_lens = np.zeros((T, K_TELOP), dtype=np.float32)
        telop_mask = torch.zeros(T, K_TELOP, dtype=torch.bool)
        stt_slots = torch.zeros(T, K_STT, SLOT_DIM)
        stt_mask = torch.zeros(T, K_STT, dtype=torch.bool)
        gt_masks = np.zeros((T, GRID_H, GRID_W), dtype=np.float32)

        for t in range(T):
            t_sec = t * BIN_SIZE

            # 활성 텔롭
            active = (inst_starts <= t_sec) & (inst_ends > t_sec)
            active_idx = np.where(active)[0]
            n_active = len(active_idx)

            scalar_feats[t, 0] = t / max(T - 1, 1)
            scalar_feats[t, 1] = n_active / 20.0

            for j, ai in enumerate(active_idx[:K_TELOP]):
                telop_slots[t, j] = inst_embs_proj[ai]
                telop_lens[t, j] = inst_tlens[ai] / 200.0
                telop_mask[t, j] = True

            # 활성 STT
            if n_stt > 0:
                stt_active = (stt_starts <= t_sec) & (stt_ends > t_sec)
                stt_active_idx = np.where(stt_active)[0]
                scalar_feats[t, 2] = len(stt_active_idx) / 5.0
                for j, si in enumerate(stt_active_idx[:K_STT]):
                    stt_slots[t, j] = stt_embs_proj[si]
                    stt_mask[t, j] = True

            # GT mask (중심좌표 → bbox)
            for ai in active_idx:
                cx, cy = inst_cx[ai], inst_cy[ai]
                w, h = inst_w[ai], inst_h[ai]
                x0 = int(np.clip(round(cx - w / 2), 0, GRID_W - 1))
                y0 = int(np.clip(round(cy - h / 2), 0, GRID_H - 1))
                x1 = int(np.clip(round(cx + w / 2), 1, GRID_W))
                y1 = int(np.clip(round(cy + h / 2), 1, GRID_H))
                gt_masks[t, y0:y1, x0:x1] = 1.0

        return {
            "channel_id": torch.tensor(channel_id, dtype=torch.long),
            "channel_emb": channel_emb,                              # (EMB_DIM,)
            "scalar_feats": torch.from_numpy(scalar_feats),           # (T, 3)
            "telop_slots": telop_slots,                               # (T, K_TELOP, SLOT_DIM)
            "telop_lens": torch.from_numpy(telop_lens),               # (T, K_TELOP)
            "telop_mask": telop_mask,                                 # (T, K_TELOP)
            "stt_slots": stt_slots,                                   # (T, K_STT, SLOT_DIM)
            "stt_mask": stt_mask,                                     # (T, K_STT)
            "gt_masks": torch.from_numpy(gt_masks),                   # (T, 80, 80)
            "T": T,
        }


def collate_fn(batch):
    max_T = max(b["T"] for b in batch)
    B = len(batch)

    channel_ids = torch.zeros(B, dtype=torch.long)
    channel_embs = torch.zeros(B, EMB_DIM)
    scalar_feats = torch.zeros(B, max_T, 3)
    telop_slots = torch.zeros(B, max_T, K_TELOP, SLOT_DIM)
    telop_lens = torch.zeros(B, max_T, K_TELOP)
    telop_mask = torch.zeros(B, max_T, K_TELOP, dtype=torch.bool)
    stt_slots = torch.zeros(B, max_T, K_STT, SLOT_DIM)
    stt_mask = torch.zeros(B, max_T, K_STT, dtype=torch.bool)
    gt_masks = torch.zeros(B, max_T, GRID_H, GRID_W)
    time_mask = torch.zeros(B, max_T, dtype=torch.bool)

    for i, b in enumerate(batch):
        T = b["T"]
        channel_ids[i] = b["channel_id"]
        channel_embs[i] = b["channel_emb"]
        scalar_feats[i, :T] = b["scalar_feats"]
        telop_slots[i, :T] = b["telop_slots"]
        telop_lens[i, :T] = b["telop_lens"]
        telop_mask[i, :T] = b["telop_mask"]
        stt_slots[i, :T] = b["stt_slots"]
        stt_mask[i, :T] = b["stt_mask"]
        gt_masks[i, :T] = b["gt_masks"]
        time_mask[i, :T] = True

    return {
        "channel_id": channel_ids,
        "channel_emb": channel_embs,
        "scalar_feats": scalar_feats,
        "telop_slots": telop_slots,
        "telop_lens": telop_lens,
        "telop_mask": telop_mask,
        "stt_slots": stt_slots,
        "stt_mask": stt_mask,
        "gt_masks": gt_masks,
        "time_mask": time_mask,
    }


train_ds = FrameMaskDataset(train_samples, channel2id, text2emb)
eval_ds = FrameMaskDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

batch = next(iter(train_loader))
print(f"✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

pos_ratio = batch['gt_masks'][batch['time_mask']].mean().item()
print(f"\n  positive 비율 (배치 근사): {pos_ratio:.4f}")

✅ 배치 확인
  channel_id: torch.Size([32])
  channel_emb: torch.Size([32, 1024])
  scalar_feats: torch.Size([32, 1734, 3])
  telop_slots: torch.Size([32, 1734, 20, 128])
  telop_lens: torch.Size([32, 1734, 20])
  telop_mask: torch.Size([32, 1734, 20])
  stt_slots: torch.Size([32, 1734, 5, 128])
  stt_mask: torch.Size([32, 1734, 5])
  gt_masks: torch.Size([32, 1734, 80, 80])
  time_mask: torch.Size([32, 1734])

  positive 비율 (배치 근사): 0.0752


In [3]:
# %% 셀 3: 모델 정의
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 6
D_FF = 512
DROPOUT = 0.1
SPATIAL_CH = 16
POS_WEIGHT = 11.8

INTRA_DIM = 64
CROSS_DIM = 64
STT_CTX_DIM = 64
CH_ID_DIM = 64
CH_TEXT_DIM = 32
SCALAR_DIM = 32
# concat: 64 + 64 + 64 + 64 + 32 + 32 = 320 → proj → 256


class IntraFrameModule(nn.Module):
    """프레임 내 텔롭 self-attention → attention pool → telop_context"""
    def __init__(self, slot_dim=SLOT_DIM, d_out=INTRA_DIM, n_heads=4):
        super().__init__()
        self.proj_in = nn.Linear(slot_dim + 1, d_out)  # +1 for text_len
        self.self_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_out))
        self.pool_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)

    def forward(self, slot_embs, slot_lens, slot_mask):
        """
        slot_embs: (BT, K, SLOT_DIM)
        slot_lens: (BT, K)
        slot_mask: (BT, K) True=유효
        Returns:   (BT, d_out)
        """
        BT, K, _ = slot_embs.shape

        # text_len concat
        x = torch.cat([slot_embs, slot_lens.unsqueeze(-1)], dim=-1)  # (BT, K, SLOT_DIM+1)
        x = self.proj_in(x)  # (BT, K, d_out)

        any_active = slot_mask.any(dim=1)  # (BT,)

        pad_mask = ~slot_mask  # True=패딩
        # 전부 패딩인 행은 풀어줌 (NaN 방지)
        all_padded = ~any_active
        pad_mask[all_padded] = False

        # self-attention
        x_att, _ = self.self_attn(x, x, x, key_padding_mask=pad_mask)

        # attention pooling
        query = self.pool_query.expand(BT, -1, -1)
        pooled, _ = self.pool_attn(query, x_att, x_att, key_padding_mask=pad_mask)
        pooled = pooled.squeeze(1)  # (BT, d_out)

        # 활성 없는 프레임은 0
        pooled = pooled * any_active.unsqueeze(-1).float()

        return pooled


class CrossAttentionModule(nn.Module):
    """텔롭(query) → STT(key/value) cross-attention → pool → cross_context"""
    def __init__(self, slot_dim=SLOT_DIM, d_out=CROSS_DIM, n_heads=4):
        super().__init__()
        self.telop_proj = nn.Linear(slot_dim + 1, d_out)
        self.stt_proj = nn.Linear(slot_dim, d_out)
        self.cross_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_out))
        self.pool_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)

    def forward(self, telop_embs, telop_lens, telop_mask, stt_embs, stt_mask):
        """
        telop_embs: (BT, K, SLOT_DIM)
        telop_lens: (BT, K)
        telop_mask: (BT, K)
        stt_embs:   (BT, S, SLOT_DIM)
        stt_mask:   (BT, S)
        Returns:    (BT, d_out)
        """
        BT = telop_embs.shape[0]

        has_telop = telop_mask.any(dim=1)  # (BT,)
        has_stt = stt_mask.any(dim=1)      # (BT,)
        both = has_telop & has_stt         # (BT,)

        q = torch.cat([telop_embs, telop_lens.unsqueeze(-1)], dim=-1)
        q = self.telop_proj(q)  # (BT, K, d_out)
        kv = self.stt_proj(stt_embs)  # (BT, S, d_out)

        stt_pad = ~stt_mask
        # 둘 다 없는 프레임은 NaN 방지로 풀어줌
        stt_pad[~both] = False

        cross_out, _ = self.cross_attn(q, kv, kv, key_padding_mask=stt_pad)  # (BT, K, d_out)

        # pool cross output
        telop_pad = ~telop_mask
        telop_pad[~both] = False

        query = self.pool_query.expand(BT, -1, -1)
        pooled, _ = self.pool_attn(query, cross_out, cross_out, key_padding_mask=telop_pad)
        pooled = pooled.squeeze(1)  # (BT, d_out)

        # 둘 다 있는 프레임만 유효
        pooled = pooled * both.unsqueeze(-1).float()

        return pooled


class STTContextModule(nn.Module):
    """STT slots → attention pool → stt_context"""
    def __init__(self, slot_dim=SLOT_DIM, d_out=STT_CTX_DIM, n_heads=4):
        super().__init__()
        self.proj_in = nn.Linear(slot_dim, d_out)
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_out))
        self.pool_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)

    def forward(self, stt_embs, stt_mask):
        """
        stt_embs: (BT, S, SLOT_DIM)
        stt_mask: (BT, S) True=유효
        Returns:  (BT, d_out)
        """
        BT = stt_embs.shape[0]

        x = self.proj_in(stt_embs)  # (BT, S, d_out)

        has_stt = stt_mask.any(dim=1)  # (BT,)

        pad_mask = ~stt_mask
        pad_mask[~has_stt] = False

        query = self.pool_query.expand(BT, -1, -1)
        pooled, _ = self.pool_attn(query, x, x, key_padding_mask=pad_mask)
        pooled = pooled.squeeze(1)

        pooled = pooled * has_stt.unsqueeze(-1).float()

        return pooled


class FrameMaskModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL,
                 n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        # 채널 인코딩
        self.channel_id_emb = nn.Embedding(n_channels, CH_ID_DIM)
        self.channel_text_proj = nn.Linear(emb_dim, CH_TEXT_DIM)

        # 스칼라 인코딩
        self.scalar_enc = nn.Sequential(
            nn.Linear(3, 32),
            nn.GELU(),
            nn.Linear(32, SCALAR_DIM),
        )

        # 프레임 내 모듈
        self.intra_telop = IntraFrameModule(SLOT_DIM, INTRA_DIM)
        self.cross_module = CrossAttentionModule(SLOT_DIM, CROSS_DIM)
        self.stt_context = STTContextModule(SLOT_DIM, STT_CTX_DIM)

        # concat → d_model
        concat_dim = CH_ID_DIM + CH_TEXT_DIM + SCALAR_DIM + INTRA_DIM + CROSS_DIM + STT_CTX_DIM
        self.token_proj = nn.Sequential(
            nn.Linear(concat_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        # Temporal Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.temporal_transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )

        # Spatial Decoder: d_model → (1, 80, 80)
        self.to_spatial = nn.Linear(d_model, SPATIAL_CH * 10 * 10)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(SPATIAL_CH, 8, kernel_size=4, stride=2, padding=1),  # 10→20
            nn.BatchNorm2d(8),
            nn.GELU(),
            nn.ConvTranspose2d(8, 4, kernel_size=4, stride=2, padding=1),  # 20→40
            nn.BatchNorm2d(4),
            nn.GELU(),
            nn.ConvTranspose2d(4, 1, kernel_size=4, stride=2, padding=1),  # 40→80
        )

    def forward(self, channel_id, channel_emb, scalar_feats,
                telop_slots, telop_lens, telop_mask,
                stt_slots, stt_mask, time_mask):
        B, T, _ = scalar_feats.shape

        # ① 채널
        ch_id = self.channel_id_emb(channel_id).unsqueeze(1).expand(-1, T, -1)
        ch_text = self.channel_text_proj(channel_emb).unsqueeze(1).expand(-1, T, -1)

        # ② 스칼라
        scalar = self.scalar_enc(scalar_feats)

        # ③ 프레임 내 모듈 (BT 단위로 flatten)
        telop_flat = telop_slots.view(B * T, K_TELOP, SLOT_DIM)
        telop_lens_flat = telop_lens.view(B * T, K_TELOP)
        telop_mask_flat = telop_mask.view(B * T, K_TELOP)
        stt_flat = stt_slots.view(B * T, K_STT, SLOT_DIM)
        stt_mask_flat = stt_mask.view(B * T, K_STT)

        telop_ctx = self.intra_telop(telop_flat, telop_lens_flat, telop_mask_flat)  # (BT, 64)
        cross_ctx = self.cross_module(telop_flat, telop_lens_flat, telop_mask_flat,
                                      stt_flat, stt_mask_flat)                       # (BT, 64)
        stt_ctx = self.stt_context(stt_flat, stt_mask_flat)                          # (BT, 64)

        telop_ctx = telop_ctx.view(B, T, INTRA_DIM)
        cross_ctx = cross_ctx.view(B, T, CROSS_DIM)
        stt_ctx = stt_ctx.view(B, T, STT_CTX_DIM)

        # ④ concat → proj
        concat = torch.cat([ch_id, ch_text, scalar, telop_ctx, cross_ctx, stt_ctx], dim=-1)
        tokens = self.token_proj(concat)

        # ⑤ Temporal Transformer
        padding_mask = ~time_mask
        out = self.temporal_transformer(tokens, src_key_padding_mask=padding_mask)

        # ⑥ Spatial Decoder
        spatial_flat = self.to_spatial(out)
        spatial = spatial_flat.view(B * T, SPATIAL_CH, 10, 10)
        masks = self.decoder(spatial)
        masks = masks.view(B, T, GRID_H, GRID_W)

        return masks


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FrameMaskModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")

🖥️  Device: cuda
📐 파라미터: 3,852,197


In [ ]:
# %% 셀 4: 학습
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
SAVE_DIR = "./model/8_layout_frame_mask_attn"
os.makedirs(SAVE_DIR, exist_ok=True)


def compute_loss(pred, gt, time_mask):
    mask_3d = time_mask.unsqueeze(-1).unsqueeze(-1).expand_as(pred)
    pred_valid = pred[mask_3d]
    gt_valid = gt[mask_3d]
    pos_weight = torch.tensor([POS_WEIGHT], device=pred.device)
    return F.binary_cross_entropy_with_logits(pred_valid, gt_valid, pos_weight=pos_weight)


@torch.no_grad()
def compute_metrics(pred, gt, time_mask, threshold=0.5):
    mask_3d = time_mask.unsqueeze(-1).unsqueeze(-1).expand_as(pred)
    pred_bin = (torch.sigmoid(pred[mask_3d]) > threshold).float()
    gt_valid = gt[mask_3d]

    tp = (pred_bin * gt_valid).sum().item()
    fp = (pred_bin * (1 - gt_valid)).sum().item()
    fn = ((1 - pred_bin) * gt_valid).sum().item()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    return {"precision": precision, "recall": recall, "f1": f1}


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0, 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_id = batch["channel_id"].to(device)
        channel_emb = batch["channel_emb"].to(device)
        scalar_feats = batch["scalar_feats"].to(device)
        telop_slots = batch["telop_slots"].to(device)
        telop_lens = batch["telop_lens"].to(device)
        telop_mask = batch["telop_mask"].to(device)
        stt_slots = batch["stt_slots"].to(device)
        stt_mask = batch["stt_mask"].to(device)
        gt_masks = batch["gt_masks"].to(device)
        time_mask = batch["time_mask"].to(device)

        pred = model(channel_id, channel_emb, scalar_feats,
                     telop_slots, telop_lens, telop_mask,
                     stt_slots, stt_mask, time_mask)
        loss = compute_loss(pred, gt_masks, time_mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    model.eval()
    eval_loss_sum, eval_batches = 0, 0
    all_metrics = {"precision": [], "recall": [], "f1": []}

    with torch.no_grad():
        for batch in eval_loader:
            channel_id = batch["channel_id"].to(device)
            channel_emb = batch["channel_emb"].to(device)
            scalar_feats = batch["scalar_feats"].to(device)
            telop_slots = batch["telop_slots"].to(device)
            telop_lens = batch["telop_lens"].to(device)
            telop_mask = batch["telop_mask"].to(device)
            stt_slots = batch["stt_slots"].to(device)
            stt_mask = batch["stt_mask"].to(device)
            gt_masks = batch["gt_masks"].to(device)
            time_mask = batch["time_mask"].to(device)

            pred = model(channel_id, channel_emb, scalar_feats,
                         telop_slots, telop_lens, telop_mask,
                         stt_slots, stt_mask, time_mask)
            loss = compute_loss(pred, gt_masks, time_mask)
            metrics = compute_metrics(pred, gt_masks, time_mask)

            eval_loss_sum += loss.item()
            eval_batches += 1
            for k, v in metrics.items():
                all_metrics[k].append(v)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss = eval_loss_sum / max(eval_batches, 1)
    avg_m = {k: np.mean(v) for k, v in all_metrics.items()}
    lr_now = optimizer.param_groups[0]["lr"]

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"P={avg_m['precision']:.3f}  R={avg_m['recall']:.3f}  F1={avg_m['f1']:.3f}  "
        f"lr={lr_now:.2e}"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

[1/50] train:   4%|▍         | 70/1768 [01:19<23:44,  1.19it/s]  